In [ ]:
import sys
sys.path.append("D:/cxr-triage")
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve
from tqdm import tqdm
from src.data.dataset import ChestXrayDataset
from src.data.transforms import get_val_transforms
from src.models.convnext import ConvNeXtModel

LABELS = ["Atelectasis","Consolidation","Infiltration","Pneumothorax","Edema","Emphysema","Fibrosis","Effusion","Pneumonia","Pleural_Thickening","Cardiomegaly","Nodule","Mass","Hernia"]
CRITICAL = ["Pneumothorax","Edema","Pneumonia","Hernia"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_ROOT = "F:/X ray dataset/Second Version"
CHECKPOINT_PATH = "D:/cxr-triage/checkpoints/convnext_focal_fixed/best_model.pth"
IMAGE_SIZE = 224
USE_CLAHE = False
MODEL_LABEL = "ConvNeXt-Tiny + Focal Loss (fixed pipeline)"

model = ConvNeXtModel(num_classes=14, pretrained=False).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Device: {DEVICE}")
print(f"Evaluating: {MODEL_LABEL}")
print(f"Checkpoint epoch: {checkpoint[chr(39)]epoch[chr(39)]+1}")
print(f"Checkpoint best AUC: {checkpoint[chr(39)]best_auc[chr(39)]:.4f}")


In [ ]:
train_df = pd.read_csv("D:/cxr-triage/data/processed/train.csv")
val_df = pd.read_csv("D:/cxr-triage/data/processed/val.csv")
test_df = pd.read_csv("D:/cxr-triage/data/processed/test.csv")
train_p = set(train_df["Patient ID"])
val_p = set(val_df["Patient ID"])
test_p = set(test_df["Patient ID"])
tv,tt,vt = len(train_p&val_p),len(train_p&test_p),len(val_p&test_p)
print(f"Train-Val:{tv} Train-Test:{tt} Val-Test:{vt}")
assert tv==0 and tt==0 and vt==0
print("PASS: zero leakage")
assert LABELS==["Atelectasis","Consolidation","Infiltration","Pneumothorax","Edema","Emphysema","Fibrosis","Effusion","Pneumonia","Pleural_Thickening","Cardiomegaly","Nodule","Mass","Hernia"]
print("PASS: label order correct")
assert isinstance(model.model.classifier[2], torch.nn.Linear)
print("PASS: no sigmoid in classifier")


In [ ]:
def get_predictions(df, batch_size=16):
    transform = get_val_transforms(image_size=IMAGE_SIZE, use_clahe=USE_CLAHE)
    dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=transform)
    dataset.df = df
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    all_logits, all_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Inference"):
            images = images.to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(images)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.numpy())
    logits = np.concatenate(all_logits,axis=0).astype(np.float32)
    targets = np.concatenate(all_targets,axis=0).astype(np.float32)
    probs = 1/(1+np.exp(-logits))
    return probs, targets
print("Inference function ready")


In [ ]:
val_probs, val_targets = get_predictions(val_df)
print(f"Val shape: {val_probs.shape}")
optimal_thresholds = {}
print(f"{chr(39)}Finding{chr(39):<22} {chr(39)}Threshold{chr(39):<12} {chr(39)}Val F1{chr(39):<10}")
print("-"*50)
for i, label in enumerate(LABELS):
    precisions,recalls,thresh = precision_recall_curve(val_targets[:,i],val_probs[:,i])
    f1s = 2*precisions[:-1]*recalls[:-1]/(precisions[:-1]+recalls[:-1]+1e-8)
    best_idx = np.argmax(f1s)
    optimal_thresholds[label] = float(thresh[best_idx])
    print(f"{label:<22} {thresh[best_idx]:<12.4f} {f1s[best_idx]:<10.4f}")
print("Thresholds frozen from validation set only.")


In [ ]:
test_probs, test_targets = get_predictions(test_df)
test_pred = np.zeros_like(test_probs, dtype=np.float32)
for i, label in enumerate(LABELS):
    test_pred[:,i] = (test_probs[:,i]>=optimal_thresholds[label]).astype(np.float32)
triage_map = {"Pneumothorax":"Critical","Edema":"Critical","Pneumonia":"Critical","Hernia":"Critical","Consolidation":"Urgent","Effusion":"Urgent","Cardiomegaly":"Urgent","Mass":"Urgent","Atelectasis":"Urgent","Nodule":"Urgent","Infiltration":"Urgent","Emphysema":"Routine","Fibrosis":"Routine","Pleural_Thickening":"Routine"}
print(f"{chr(39)}Finding{chr(39):<22} {chr(39)}Tier{chr(39):<10} {chr(39)}AUC{chr(39):<8} {chr(39)}F1{chr(39):<8} {chr(39)}Precision{chr(39):<12} {chr(39)}Recall{chr(39):<8} {chr(39)}Support{chr(39):<8}")
print("-"*85)
per_class_auc = []
for i, label in enumerate(LABELS):
    auc = roc_auc_score(test_targets[:,i],test_probs[:,i])
    per_class_auc.append(auc)
    f1 = f1_score(test_targets[:,i],test_pred[:,i],zero_division=0)
    prec = precision_score(test_targets[:,i],test_pred[:,i],zero_division=0)
    rec = recall_score(test_targets[:,i],test_pred[:,i],zero_division=0)
    support = int(test_targets[:,i].sum())
    print(f"{label:<22} {triage_map[label]:<10} {auc:<8.4f} {f1:<8.4f} {prec:<12.4f} {rec:<8.4f} {support:<8}")
print("-"*85)
mean_auc = np.mean(per_class_auc)
critical_auc = np.mean([auc for label,auc in zip(LABELS,per_class_auc) if label in CRITICAL])
micro_f1 = f1_score(test_targets,test_pred,average="micro")
macro_f1 = f1_score(test_targets,test_pred,average="macro")
print(f"Model: {MODEL_LABEL}")
print(f"Mean Test AUC:     {mean_auc:.4f}")
print(f"Critical Test AUC: {critical_auc:.4f}")
print(f"Micro F1:          {micro_f1:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")


In [ ]:
import json
results = {"model":MODEL_LABEL,"checkpoint":CHECKPOINT_PATH,"checkpoint_epoch":checkpoint["epoch"]+1,"mean_test_auc":float(mean_auc),"critical_test_auc":float(critical_auc),"micro_f1":float(micro_f1),"macro_f1":float(macro_f1),"thresholds":optimal_thresholds}
with open("D:/cxr-triage/notebooks/convnext_focal_final_results.json","w") as f:
    json.dump(results,f,indent=2)
print("Saved to convnext_focal_final_results.json")
